# Vitiligo | SD 2.1 morphology-controlled inference

**Purpose:** Produce five synthetic full-face vitiligo candidates per prompt using SD 2.1 and **your own** facial-vitiligo LoRA. There is no included pre-trained vitiligo disease adapter. The LoRA must first be trained with Notebook 02 or otherwise obtained legitimately.

**Compel 2.3.1:** 150 SD2.1 tokenizer content tokens maximum, two 77-position CLIP chunks, no silent truncation, positive/negative embedding padding.

**Requirements:** Kaggle T4 GPU and Internet enabled for model download. Models are research-only and synthetic results are not medically verified. This notebook has not had an end-to-end GPU run in the local CPU sandbox.

## 1. Install exact library versions
Use Kaggle's pre-installed GPU-enabled PyTorch. If you have imported incompatible packages already, restart the Kaggle session after installation.

In [ ]:
import sys, subprocess
PACKAGES = [
    'diffusers==0.35.1', 'transformers==4.56.2',
    'accelerate==1.10.1', 'peft==0.17.1',
    'huggingface_hub==0.34.4', 'safetensors==0.6.2',
    'ftfy==6.3.1', 'compel==2.3.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('Dependencies installed. If imports fail due to already imported packages, restart and run from top.')

In [ ]:
import json, time, re, zipfile
from pathlib import Path
from IPython.display import display
import torch
import diffusers, transformers, accelerate
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from compel import Compel, ReturnedEmbeddingsType
from huggingface_hub import model_info
assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook Settings (T4 or another supported CUDA GPU).'
assert torch.cuda.get_device_capability(0)[0] >= 7, 'FP16 CUDA GPU required.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
print('Diffusers:', diffusers.__version__, '| Transformers:', transformers.__version__)

## 2. Configuration
Change `LORA_SOURCE` to your trained LoRA output directory or a `.safetensors` file after Notebook 02. Leave blank only for the base-model diagnostic. The SD2.1 mirror is community-maintained: verify weight provenance/licence before publication.

In [ ]:
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base'
LORA_SOURCE = ''   # example '/kaggle/input/my-vitiligo-lora/pytorch_lora_weights.safetensors'
LORA_SCALE = 0.8
OUT_DIR = Path('/kaggle/working/vitiligo_generated')
IMAGES_PER_PROMPT = 5
WIDTH = HEIGHT = 512
STEPS = 30
GUIDANCE = 7.0
BASE_SEED = 42031
MAX_CONTENT_TOKENS = 150
MAX_CLIP_CHUNKS = 2
NEGATIVE_PROMPT = ('cartoon, illustration, painting, overly smoothed skin, makeup mask, '
                   'plastic texture, blurry, warped face, duplicate face, text, watermark')
assert IMAGES_PER_PROMPT >= 1 and WIDTH % 8 == 0 and HEIGHT % 8 == 0
assert 0 <= LORA_SCALE <= 1.5
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Authored morphology prompts
Uses the project `prompts/` JSON files, 300 per class with protected group-disjoint train/validation/test prompt families. Defaults to a five-prompt **train-only** pilot and uses each prompt's original negative text. Change `PROMPT_LIMIT=None` to run all 210 train prompts. Outputs include a review-draft CSV; all generated images begin **unapproved**.


In [ ]:
# Split-safe authored prompts (210 train, 45 val, 45 test; 60 disjoint families).
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prompt_collection,generation_manifest,append_run_log
PROMPT_PARTITION='train'
PROMPT_VARIANT='morphology'  # 'generic' is E1, 'morphology' is E2  # DO NOT augment with val/test prompt families
PROMPT_OFFSET=0
PROMPT_LIMIT=5  # Kaggle pilot; None for all 210 training prompts
PROMPTS=prompt_collection(DERMSYNREAL_REPO,'Vitiligo',partition=PROMPT_PARTITION,limit=PROMPT_LIMIT,offset=PROMPT_OFFSET,variant=PROMPT_VARIANT)
print('Authored prompts:',len(PROMPTS),'families:',len({p['prompt_family_id'] for p in PROMPTS}))

## 4. Load stable diffusion base and your optional custom LoRA
Model downloads use the exact commit reported by Hugging Face. Inference uses FP16 for Kaggle T4 and does not require BF16.

In [ ]:
base_commit = model_info(BASE_MODEL).sha
print('Base:', BASE_MODEL, '| commit:', base_commit)
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, revision=base_commit, torch_dtype=torch.float16,
    use_safetensors=True,
)
if str(LORA_SOURCE).strip():
    source = str(LORA_SOURCE).strip()
    source_file = Path(source)
    if source_file.is_file():
        if source_file.suffix != '.safetensors':
            raise ValueError('LoRA file must be .safetensors')
        pipe.load_lora_weights(str(source_file.parent), weight_name=source_file.name, adapter_name='facial_vitiligo')
    else:
        pipe.load_lora_weights(source, adapter_name='facial_vitiligo')
    pipe.set_adapters(['facial_vitiligo'], adapter_weights=[LORA_SCALE])
    print('Custom facial-vitiligo LoRA enabled')
else:
    print('Base-only diagnostic: no custom LoRA loaded.')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.to('cuda')
pipe.text_encoder.eval()
pipe.set_progress_bar_config(disable=False)

## 5. Compel: SD2.1 150-content-token mode
SD2.1 uses 77 text positions per chunk: 75 ordinary content tokens plus two boundary tokens. Validate **both** the positive and negative strings with the tokenizer of the loaded model before sending either to Compel. Compel does not truncate; outputs must fit two CLIP chunks. This is an **inference-only** feature: the official LoRA trainer still truncates its training captions to one CLIP chunk.

In [ ]:
tokenizer = pipe.tokenizer
assert tokenizer.model_max_length == 77, f'Unexpected tokenizer context: {tokenizer.model_max_length}'
CONTENT_PER_CHUNK = tokenizer.model_max_length - 2
assert MAX_CONTENT_TOKENS == 150 and MAX_CLIP_CHUNKS == 2
assert MAX_CONTENT_TOKENS <= CONTENT_PER_CHUNK * MAX_CLIP_CHUNKS

# For SD2.1 use last normalized hidden states; SDXL's pooled/penultimate setup is not applicable.
compel = Compel(
    tokenizer=pipe.tokenizer,
    text_encoder=pipe.text_encoder,
    returned_embeddings_type=ReturnedEmbeddingsType.LAST_HIDDEN_STATES_NORMALIZED,
    truncate_long_prompts=False,
    requires_pooled=False,
    device='cuda',
)

def content_token_count(text):
    # The exact tokenizer instance used by SD2.1; no length truncation.
    ids = tokenizer(text, truncation=False, add_special_tokens=False)['input_ids']
    return len(ids)

def validate_prompt(text, label):
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f'{label}: prompt must be non-empty')
    num = content_token_count(text)
    if num > MAX_CONTENT_TOKENS:
        raise ValueError(f'{label}: {num} content tokens exceeds {MAX_CONTENT_TOKENS}; shorten the prompt')
    print(f'{label}: {num} content tokens, {(num + CONTENT_PER_CHUNK-1)//CONTENT_PER_CHUNK or 1} CLIP chunk(s)')
    return num

# Fail BEFORE any generation.
for item in PROMPTS:
    validate_prompt(item['text'], item['id'])
for item in PROMPTS:
    validate_prompt(item.get('negative_prompt',NEGATIVE_PROMPT),item['id']+'/negative')

## 6. Generate five images per prompt
Compel encodes positive and negative prompts separately, pads them to equal length, and passes embeddings directly into the diffusion pipeline. Files receive deterministic seeds and metadata.

In [ ]:
def safe_id(s):
    return re.sub(r'[^a-zA-Z0-9_-]+', '_', s).strip('_')[:72] or 'sample'

metadata = []
started = time.time()
for i, item in enumerate(PROMPTS):
    positive_text = item['text'].strip()
    validate_prompt(positive_text, item['id'])
    with torch.no_grad():
        positive_embeddings = compel(positive_text)
        negative_embeddings = compel(item.get('negative_prompt',NEGATIVE_PROMPT))
        positive_embeddings, negative_embeddings = compel.pad_conditioning_tensors_to_same_length(
            [positive_embeddings, negative_embeddings]
        )
    assert positive_embeddings.shape == negative_embeddings.shape, 'Conditioning shape mismatch'
    assert positive_embeddings.ndim == 3
    assert positive_embeddings.shape[1] <= 77 * MAX_CLIP_CHUNKS, 'Compel produced more than two chunks'
    positive_embeddings = positive_embeddings.to(device='cuda', dtype=pipe.unet.dtype)
    negative_embeddings = negative_embeddings.to(device='cuda', dtype=pipe.unet.dtype)
    images = []
    folder = OUT_DIR / safe_id(item['id'])
    folder.mkdir(parents=True, exist_ok=True)
    for j in range(IMAGES_PER_PROMPT):
        seed = BASE_SEED + i * 1000 + j
        generator = torch.Generator(device='cuda').manual_seed(seed)
        with torch.inference_mode():
            image = pipe(
                prompt_embeds=positive_embeddings,
                negative_prompt_embeds=negative_embeddings,
                height=HEIGHT, width=WIDTH, num_inference_steps=STEPS,
                guidance_scale=GUIDANCE, generator=generator,
            ).images[0]
        name = f'{safe_id(item["id"])}_seed{seed}.png'
        fp = folder / name
        image.save(fp)
        images.append(image)
        metadata.append({
            'file': fp.relative_to(OUT_DIR).as_posix(),
            'prompt_id': item['id'], 'pattern_requested': item.get('pattern'),
            'distribution_requested': item.get('distribution'),
            'prompt': positive_text, 'negative_prompt': item.get('negative_prompt',NEGATIVE_PROMPT),
            'positive_content_tokens': content_token_count(positive_text),
            'negative_content_tokens': content_token_count(item.get('negative_prompt',NEGATIVE_PROMPT)),
            'clip_embedding_positions': int(positive_embeddings.shape[1]),
            'seed': seed, 'base': BASE_MODEL, 'base_revision': base_commit,
            'lora_source': LORA_SOURCE or None,
            'lora_scale': LORA_SCALE if LORA_SOURCE else None,
            'steps': STEPS, 'guidance': GUIDANCE,
            'width': WIDTH, 'height': HEIGHT,
            'clinical_review': 'pending',
        })
    preview = Image.new('RGB', (WIDTH * len(images), HEIGHT))
    for k, image in enumerate(images):
        preview.paste(image, (k * WIDTH, 0))
    out_width = min(1250, preview.width)
    display(preview.resize((out_width, int(preview.height * out_width / preview.width))))

with (OUT_DIR / 'metadata.jsonl').open('w', encoding='utf-8') as file:
    for item in metadata:
        file.write(json.dumps(item, ensure_ascii=False) + '\n')
print('Saved:', len(metadata), 'images, seconds:', round(time.time() - started, 1))

In [ ]:
# Write Kaggle CSV/JSON audit outputs BEFORE packaging the generated images.
summary=generation_manifest(metadata,OUT_DIR/'reports',root=OUT_DIR,class_name='Vitiligo',repo_root=DERMSYNREAL_REPO,prompt_partition=PROMPT_PARTITION,model_id=BASE_MODEL,prompt_type=PROMPT_VARIANT)
append_run_log(OUT_DIR,kind='generation',config={'class':'Vitiligo','prompt_partition':PROMPT_PARTITION,'prompt_type':PROMPT_VARIANT,'prompt_count':len(PROMPTS),'images_per_prompt':IMAGES_PER_PROMPT,'generated':summary['generated_images']},artifacts=[OUT_DIR/'reports/generation_manifest.csv',OUT_DIR/'metadata.jsonl'],repo_root=DERMSYNREAL_REPO)
print('Review draft:',OUT_DIR/'reports/synthetic_pool_review_draft.csv','| Run log:',OUT_DIR/'run_log.jsonl')

## 7. ZIP images and metadata
Download the resulting ZIP from Kaggle **Output**. Generated images are unreviewed; please run morphology and privacy duplicate checks before classifier training.

In [ ]:
zip_path = Path('/kaggle/working/vitiligo_generated.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for fp in sorted(OUT_DIR.rglob('*')):
        if fp.is_file():
            archive.write(fp, fp.relative_to(OUT_DIR).as_posix())
assert len(metadata) == len(PROMPTS) * IMAGES_PER_PROMPT
print('ZIP:', zip_path, '| size:', round(zip_path.stat().st_size / (1024**2), 2), 'MiB')

## Sources and limitations
- [SD2.1 community mirror](https://huggingface.co/sd2-community/stable-diffusion-2-1-base)
- [Compel 2.3.1](https://pypi.org/project/compel/2.3.1/)
- [Diffusers 0.35.1 LoRA](https://huggingface.co/docs/diffusers/v0.35.1/en/training/lora)

Full GPU and checkpoint download verification cannot be performed in a CPU-only offline environment.